# Energy Bills Automation

## Version control

In [28]:
import os
from pathlib import Path
from dotenv import dotenv_values

if os.getenv("POSIT_PRODUCT") == "WORKBENCH":
    token = dotenv_values(f"{Path(os.getenv('HOME'))}/dev.env")["git_token"]
    
    !rm -rf .git  # start clean git commit
    !git init
    !git add .
    !git commit -m "Initial Energy Bills project commit"
    !git branch -M main
    !git config --global user.name "StathZa"
    !git config --global user.email "142407371+StathZa@users.noreply.github.com"
    
    !git remote set-url origin https://{token}@github.com/StathZa/OTE.git
    
    !git push -u origin main

Reinitialized existing Git repository in /home/eyzacharis/Energy Bills/.git/
[main 814bd94] Initial Energy Bills project commit
 2 files changed, 41 insertions(+), 81 deletions(-)
 mode change 100644 => 100755 .gitignore
Enumerating objects: 55, done.
Counting objects: 100% (55/55), done.
Delta compression using up to 24 threads
Compressing objects: 100% (53/53), done.
Writing objects: 100% (55/55), 546.24 KiB | 2.09 MiB/s, done.
Total 55 (delta 17), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (17/17), done.
remote: error: GH013: Repository rule violations found for refs/heads/main.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-pus

## Initialise Project 

In [1]:
import os 

if os.getenv("POSIT_PRODUCT") == "WORKBENCH":
    import re, shutil
    
    # Step 0: initialise project with uv
    !uv init

    # Step 1: Ensure uv is installed
    if shutil.which("uv") is None:
        print("Installing uv...")
        !pip install uv
        !uv init
    else:
        print(f"uv already available at: {shutil.which('uv')}")

    # Step 2: Ensure pipreqs is available
    if shutil.which("pipreqs") is None:
        print("Adding pipreqs via uv...")
        !uv add pipreqs
    else:
        print(f"pipreqs already available at: {shutil.which('pipreqs')}")

    # Step 3: Generate requirements and resolve packages incompatibility
    !uvx pipreqs . --force --encoding=utf-8 --ignore .venv,__pycache__,.git
    !sed -i 's/numpy==.*/numpy==1.26.4/' "requirements.txt"
    !sed -i 's/pandas==.*/pandas==2.3.3/' "requirements.txt"
    !sed -i 's/python-dotenv==.*/python-dotenv==1.0.1/' "requirements.txt"

    # Step 4: Install packages from requirements.txt
    !uv python pin 3.9
    !uv add -r requirements.txt

error: Project is already initialized in `/home/eyzacharis/Energy Bills` (`pyproject.toml` file exists)
uv already available at: /home/eyzacharis/.local/bin/uv
Adding pipreqs via uv...
Resolved 83 packages in 5ms
Checked 65 packages in 5ms
⠙ pyzmq==27.1.0                                                                 INFO: Not scanning for jupyter notebooks.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements 

## Import Libraries and Utilities

In [2]:
#!/usr/bin/env python3
from utils.logger import BaseLogger
from utils.dependencies import *
from utils.data_process import _dtypes_convert, load_creds
from utils.table_reader import *
from utils.queries import pollaploi_query, parohes_cosmote_query 
from utils.type_mapping import type_mapping
from utils.profiler import PipelineProfiler
from utils.send_mail import *

In [3]:
logger = BaseLogger().logger
prof = PipelineProfiler(logger)

In [4]:
conn_info = load_creds(logger=logger)

15-05-2026 14:19:12 INFO [data_process.load_creds:18]: Loaded credentials from environment file


## Import Data

In [6]:
logger.info(f"Logging started at {datetime.now()}")

n_workers = 60
logger.info(f"Preparing to load tables with deault number of concurrent workers: {n_workers}")

# Read tables
logger.info("Reading table: energy_efficiency.pollaploi")
with prof.stage("fetch_pollaploi"):
    df = read_vertica_table_with_multiprocessing(conn_info, pollaploi_query, num_workers=n_workers, logger=logger)

logger.info("Reading table: energy_efficiency.parohes_cosmote")
with prof.stage("fetch_parohes"):
    sites = read_vertica_table(conn_info, parohes_cosmote_query)

# Apply basic data processing
logger.info("Processing data: filling N/A values, applying type casting, and creating required columns.")

15-05-2026 14:20:26 INFO [2911747232.<module>:1]: Logging started at 2026-05-15 14:20:26.562676
15-05-2026 14:20:26 INFO [2911747232.<module>:4]: Preparing to load tables with deault number of concurrent workers: 60
15-05-2026 14:20:26 INFO [2911747232.<module>:7]: Reading table: energy_efficiency.pollaploi
15-05-2026 14:20:26 WARNING [table_reader.read_vertica_table_with_multiprocessing:28]: Invalid number of workers passed. Reverting to permissible range.
15-05-2026 14:20:26 INFO [table_reader.read_vertica_table_with_multiprocessing:31]: 
    SELECT
        *
    FROM  energy_efficiency.pollaploi
    WHERE MOD(HASH(paroxos, paroxi, etos), 24) = 

15-05-2026 14:24:13 INFO [profiler._log:49]: [profiler] OK fetch_pollaploi      wall=222878.4ms  cpu=139443.1ms  peak=7832.0MB
15-05-2026 14:24:13 INFO [2911747232.<module>:11]: Reading table: energy_efficiency.parohes_cosmote
15-05-2026 14:24:14 INFO [profiler._log:49]: [profiler] OK fetch_parohes        wall=   251.8ms  cpu=   236.8ms  pea

In [7]:
with prof.stage("types_conversion"):
    df, enanti, ektaktoi = _dtypes_convert(df=df, logger=logger)

15-05-2026 14:24:14 INFO [data_process._dtypes_convert:42]: Filling last_date & previous_date N/A values for final invoice and special bill type
15-05-2026 14:29:58 INFO [data_process._dtypes_convert:106]: Fetching 52640 ektaktoi, 1551177 enanti and 868565 Ekk or Tel
15-05-2026 14:30:02 INFO [profiler._log:49]: [profiler] OK types_conversion     wall=346061.7ms  cpu=350667.6ms  peak=7324.3MB


In [11]:
# Change 'previous_date' for Εκκαθαριστικούς & Τελικούς
df.loc[(df['previous_date'].isnull()) & (df['paroxi']==(df['paroxi'].shift(1))), "previous_date"] = df['last_date'].shift(1)
df.loc[(df['last_date'].isnull())
       & (df['previous_date'].isnull()), "previous_date"] = df["Date_Bill"] - pd.to_timedelta(30, unit='d')


logger.info("Filling last_date & previous_date N/A values for account_payment bill type")
df = pd.concat([df, enanti], ignore_index=True)


df = df.sort_values(by=["paroxi", "Date_Bill"], ascending=[True, True]).reset_index(drop=True)


df.loc[(df['last_date'].isnull()) & ((df["bill_type"] == 'Εκκαθαριστικός')
                                     | (df["bill_type"] == 'Τελικός')), "last_date"] = df["previous_date"].shift(-1)
df.loc[(df['last_date'].isnull()) & ((df["bill_type"] == 'Εκκαθαριστικός')
                                     | (df["bill_type"] == 'Τελικός')), "last_date"] = df["Date_Bill"]

df.loc[
    (df['last_date'].isnull()) &
    (df['previous_date'].notnull()) &
    ((df["bill_type"] == 'Εκκαθαριστικός') | (df["bill_type"] == 'Τελικός')),
    "last_date"
] = df['previous_date'] + pd.to_timedelta(30, unit='d')


df.loc[(df['last_date'].isnull()) & (df["bill_type"] == "Έναντι"), "last_date"] = df["Date_Bill"]

df.loc[(df['hmeromhnia_teleutaias_katametrhshs'].isnull())
       & (df['hmeromhnia_prohgoumenis_katametrhshs'].isnull())
       & (df["bill_type"] == "Έναντι")
       & (df['paroxi'].eq(df['paroxi'].shift(1))), "previous_date"] = df["last_date"].shift(1)

df.loc[(df['hmeromhnia_teleutaias_katametrhshs'].isnull())
       & (df['hmeromhnia_prohgoumenis_katametrhshs'].isnull())
       & (df["bill_type"] == "Έναντι")
       & (df['paroxi'].ne(df['paroxi'].shift(1))), "previous_date"] = df["Date_Bill"] - pd.to_timedelta(30, unit='d')

df.loc[(df['hmeromhnia_teleutaias_katametrhshs'].isnull())
       & (df['hmeromhnia_prohgoumenis_katametrhshs'] >= df['hmeromhnia_prohgoumenis_katametrhshs'].shift(1))
       & (df["bill_type"] == "Έναντι") & (df['paroxi'] == (df['paroxi'].shift(1)))
       & (df['paroxos'] == (df['paroxos'].shift(1))), "previous_date"] = df["hmeromhnia_teleutaias_katametrhshs"].shift(1)

df.loc[(df['hmeromhnia_teleutaias_katametrhshs'].isnull())
       & (df['hmeromhnia_prohgoumenis_katametrhshs'] < df['hmeromhnia_prohgoumenis_katametrhshs'].shift(1))
       & (df["bill_type"] == "Έναντι")&(df['paroxi'] == (df['paroxi'].shift(1)))
       & (df['paroxos'] == (df['paroxos'].shift(1))), "previous_date"] = df["hmeromhnia_teleutaias_katametrhshs"]

df.loc[(df['hmeromhnia_teleutaias_katametrhshs'].isnull())
       & (df['hmeromhnia_prohgoumenis_katametrhshs']>df['hmeromhnia_prohgoumenis_katametrhshs'].shift(1))
       & (df["bill_type"] == "Έναντι")
       & (df['paroxi'] == (df['paroxi'].shift(1))), "previous_date"] = df["hmeromhnia_teleutaias_katametrhshs"].shift(1)

df.loc[(df['hmeromhnia_prohgoumenis_katametrhshs'] == df['hmeromhnia_prohgoumenis_katametrhshs'].shift(1))
       & (df['hmeromhnia_teleutaias_katametrhshs'] >df ['hmeromhnia_teleutaias_katametrhshs'].shift(1))
       & (df["bill_type"] == "Έναντι") & (df['paroxi'] == (df['paroxi'].shift(1)))
       & (df['paroxos'] == df['paroxos'].shift(1))
       & (df['bill_type'] == df['bill_type'].shift(1)), "previous_date"] = df["hmeromhnia_teleutaias_katametrhshs"].shift(1)

df.loc[(df['last_date'].notnull())
        & (df['previous_date'].isnull())
        & (df["bill_type"]== "Έναντι")
        & (df['paroxi'].ne(df['paroxi'].shift(1)))
        & (df['paroxi'].ne(df['paroxi'].shift(-1))), "previous_date"] = df["last_date"] - pd.to_timedelta(30, unit='d')

df.loc[(df['last_date'].notnull())
        & (df['previous_date'].isnull())
        & (df["bill_type"]== "Έναντι")
        &(df['paroxi'].eq(df['paroxi'].shift(1))), "previous_date"] = df["last_date"].shift(1)

df.loc[(df['last_date'].notnull())
        & (df['previous_date'].isnull())
        & (df["bill_type"]== "Έναντι")
        &(df['paroxi']!=(df['paroxi'].shift(1)))
        &(df['paroxi']==(df['paroxi'].shift(-1))), "previous_date"] = df['last_date'] - pd.to_timedelta(30, unit='d')

df.loc[(df['last_date'].notnull())
        & (df['previous_date'].isnull())
        & (df["bill_type"]== "Έναντι")
        &(df['paroxi']==(df['paroxi'].shift(1)))
        &(df['paroxi']!=(df['paroxi'].shift(-1))), "previous_date"] = df['last_date'].shift(1)

df.loc[(df['last_date'].notnull())
        & (df['previous_date'].isnull())
        & (df["bill_type"]== "Έναντι")
        &(df['paroxi']==(df['paroxi'].shift(1)))
        &(df['paroxi']==(df['paroxi'].shift(-1))), "previous_date"] = df['last_date'].shift(1)

15-05-2026 11:10:32 INFO [1761516536.<module>:7]: Filling last_date & previous_date N/A values for account_payment bill type


In [12]:
# Επαναφέρουμε τους Έκτακτους στο df
df = pd.concat([df, ektaktoi])

df = df.sort_values(by=["paroxi", "Date_Bill"], ascending=[True, True]).reset_index(drop=True)

df.loc[(df['last_date'].isnull())
       & (df['previous_date'].isnull())
       & (df["bill_type"] == "Έκτακτος"), "previous_date"] = df["logistiko_date"]

df.loc[(df['last_date'].isnull())
       & (df['previous_date'].isnull())
       & (df["bill_type"] == "Έκτακτος"), "last_date"] = df["logistiko_date"]

df.loc[(df['last_date'].isnull())
       & (df['previous_date'].notnull())
       & (df["bill_type"] == "Έκτακτος"), "last_date"] = df['previous_date']

#Change

df["days"]=(df["last_date"]-df["previous_date"]).dt.days
df.loc[(df['days']<=0), "days"] = 1

df["parousa_endeiksi"]=df["parousa_endeiksi"].astype(float)
df["prohgoumeni_endeiksi"]=df["prohgoumeni_endeiksi"].fillna(0).astype(float)

df["katanalwsh_kwh"]=df["katanalwsh_kwh"].astype(float)

df["synt_wxv"]=df["synt_wxv"].astype(float)

df["energy_consumption_kwh"]=df["energy_consumption_kwh"].astype(float)

df["energy_cost"]=df["energy_cost"].astype(float)

df["number_of_days"]=df["number_of_days"].astype(float)

#fixed
df['min_metrisi'] = np.nanmin(df[['parousa_endeiksi', 'prohgoumeni_endeiksi']].values, axis=1)

df['max_metrisi'] = df[['parousa_endeiksi', 'prohgoumeni_endeiksi']].max(axis=1, skipna=True)

# correct_proigoumeni_metrisi field
df.loc[(df['parousa_endeiksi'] > df['prohgoumeni_endeiksi']), 'correct_proigoumeni_metrisi'] = df['min_metrisi']
df.loc[(df['parousa_endeiksi'] <= df['prohgoumeni_endeiksi']), 'correct_proigoumeni_metrisi'] = df['max_metrisi']

# correct_parousa_metrisi field
df['correct_parousa_metrisi']=df['max_metrisi']

df.loc[(df['parousa_endeiksi'] < df['prohgoumeni_endeiksi'])
       & (df['max_metrisi']<100000), 'correct_parousa_metrisi'] = 100000 + df['min_metrisi']

df.loc[(df['parousa_endeiksi'] < df['prohgoumeni_endeiksi'])
       & (df['max_metrisi']>100000), 'correct_parousa_metrisi'] = 1000000 + df['min_metrisi']


df['last_date']=pd.to_datetime(df['last_date'], dayfirst=True, errors='coerce')
df['previous_date']=pd.to_datetime(df['previous_date'], dayfirst=True, errors='coerce')

df["delta_days"]= df["days"]

df["delta_days"].isnull().sum(axis = 0)

0

In [13]:
# Daily Consumption Field
df.loc[(df['bill_type'] == 'Εκκαθαριστικός')
       | (df['bill_type'] == 'Τελικός'), 'daily_consumption'] = df['katanalwsh_kwh'] / df["delta_days"]
df.loc[(df['bill_type'] == 'Έναντι'), 'daily_consumption'] = df['energy_consumption_kwh'] / df["delta_days"]

# Κωδικός Τιμολογίου
df["new_bill_code"] = "Γ21"
df.loc[(df['mv_lv'] == 'MV'), 'new_bill_code'] = df['mv_lv']
df.loc[((df['neos_kodikos_timologiou'] == '22')
        | (df['neos_kodikos_timologiou'] == 'Γ22-3Φ')
        | (df['neos_kodikos_timologiou'] == 'Γ22α-3Φ')
        | (df['neos_kodikos_timologiou'] == 'Γ22α')
        | (df['neos_kodikos_timologiou'] == 'Γ22')
        | (df['neos_kodikos_timologiou'] == 'Ε22')), 'new_bill_code'] = "Γ22"

ek = df[(df["bill_type"] == "Εκκαθαριστικός") | (df["bill_type"] == "Τελικός")]

df['paroxi'] = pd.to_numeric(df['paroxi'], errors='coerce')
sites['paroxi'] = pd.to_numeric(sites['paroxi'], errors='coerce')

# checkpoint
df = pd.merge(df, sites,
              how='left',
              left_on=['paroxi'],
              right_on=['paroxi'])

df.loc[(df["category"] == "Cosmote BTS"), 'site_code_cosmote'] = df['site_code']
df.loc[(df["category"] == "Cosmote BTS"), 'onomasia'] = df['site_name']

df = df.drop(['site_code', 'site_name'], axis=1)

df.loc[(df["category"] == "Cosmote BTS"), 'siteid'] = df['site_code_cosmote']
df.loc[((df["category"] == "BUILDING") | (df["category"] == "CABIN")), 'siteid'] = df['kwdikos_eett']

In [14]:
# Coordinates
coordinates = pd.read_csv(Path(glob("**/*coordinates.csv", recursive=True)[0]))

coordinates["eett"]=coordinates["eett"].astype(int)
coordinates["eett"]=coordinates["eett"].astype(str)

coordinates=coordinates.drop_duplicates(subset=['eett'], keep='first')
coordinates=coordinates.drop_duplicates(subset=['latitude','longitude'], keep='first')

coordinates2 = pd.read_csv(Path(glob("**/*coordinates_missing.csv", recursive=True)[0]))
coordinates2["siteid"]=coordinates2["siteid"].astype(int)
coordinates2["siteid"]=coordinates2["siteid"].astype(str)
coordinates2=coordinates2.drop_duplicates(subset=['siteid'], keep='first')
coordinates2=coordinates2.drop_duplicates(subset=['latitude', 'longitude'], keep='first')

coordinates3 = pd.read_csv(Path(glob("**/*coordinates_BTS.csv", recursive=True)[0]))
coordinates3["siteid"]=coordinates3["siteid"].astype(int)
coordinates3["siteid"]=coordinates3["siteid"].astype(str)


coordinates3=coordinates3.drop_duplicates(subset=['siteid'], keep='first')

coordinates3=coordinates3.drop_duplicates(subset=['latitude','longitude'], keep='first')

coordinates = coordinates.rename(columns={'eett': 'siteid'})

coordinates = pd.concat([coordinates, coordinates2], ignore_index=True)

coordinates = pd.concat([coordinates, coordinates3], ignore_index=True)

coordinates=coordinates.drop_duplicates(subset=['siteid'], keep='first')

In [15]:
# Convert siteid to numeric safely, while keeping NaN values
df["siteid"] = pd.to_numeric(df["siteid"], errors="coerce").astype("Int64")  # Keeps NaN as <NA>, avoids conversion error
coordinates["siteid"] = pd.to_numeric(coordinates["siteid"], errors="coerce").astype("Int64")

# Convert back to string, keeping NaNs
df["siteid"] = df["siteid"].astype(str).replace("<NA>", "")
coordinates["siteid"] = coordinates["siteid"].astype(str).replace("<NA>", "")

# Trim spaces (if any)
df["siteid"] = df["siteid"].str.strip()
coordinates["siteid"] = coordinates["siteid"].str.strip()


# Check how many matching siteid values exist now
common_siteids = set(df["siteid"].unique()) & set(coordinates["siteid"].unique())

df = pd.merge(df, coordinates,
              how='left',
              on=['siteid'])

## Alerts

In [16]:
############################################## Create Alerts ##########################################################

# ##### Alert 1 - Μεγάλη Καθυστέρηση του τελευταίου Εκκαθαριστικού

logger.info("Creating Alert 1 - Μεγάλη Καθυστέρηση του τελευταίου Εκκαθαριστικού")
df = df.sort_values(by=["paroxi", "Date_Bill"], ascending=[True, True]).reset_index(drop=True)

max_date=ek[["paroxi", "last_date"]]

max_date = max_date.groupby("paroxi").max().reset_index()

max_date = max_date.rename(columns={"last_date": "last_date_ek"})

max_date['paroxi'] = max_date['paroxi'].astype(int).astype(str)
df['paroxi'] = df['paroxi'].astype(int).astype(str)

df = pd.merge(df, max_date,  how='left', left_on=['paroxi'], right_on=['paroxi'])

df.loc[(df['paroxi'] != df['paroxi'].shift(-1)) & (df["bill_type"] == "Έναντι"), 'last_enanti'] = "1"

df.loc[(df["last_enanti"] == "1")
       & (df['last_date'] != df['last_date_ek']), 'days_from_last_ek'] = (df['last_date']-df['last_date_ek']).dt.days

df.loc[(df["days_from_last_ek"] > 200), 'alert_kathisterisis'] = "1"


# ##### Alert 2 - Υψηλός Εκκαθαριστικός
logger.info("Creating Alert 2 - Υψηλός Εκκαθαριστικός")
median_consumption=ek[["paroxi","daily_consumption"]]

median_consumption=median_consumption.groupby("paroxi").median().reset_index()

median_consumption = median_consumption.rename(columns={"daily_consumption": "median_daily_consumption"})
median_consumption['paroxi'] = median_consumption['paroxi'].astype(int).astype(str)

df = pd.merge(df, median_consumption,
              how='left',
              left_on=['paroxi'],
              right_on=['paroxi'])

df["perc_consumption"]= np.abs(df["daily_consumption"] / df["median_daily_consumption"])
df["perc_consumption"]=df["perc_consumption"].astype(float)

df.loc[((df["daily_consumption"] > 0)
        & (df["median_daily_consumption"] > 0)
        & ((df["bill_type"] == "Εκκαθαριστικός")
           | (df["bill_type"] == "Τελικός"))
        & (df["perc_consumption"] > 2)
        & (df["energy_consumption_kwh"] > 0)
        & (df["correct_parousa_metrisi"]-df["correct_proigoumeni_metrisi"]>20)), 'alert_high_ekkatharistikos'] = "1"

# ##### Alert 3 - Υψηλός Έναντι
logger.info("Creating Alert 3 - Υψηλός Έναντι")
df.loc[((df["daily_consumption"] > 0)
        & (df["median_daily_consumption"] > 0)
        & (df["bill_type"] == "Έναντι")
        & (df["perc_consumption"] > 2)
        & (df["energy_consumption_kwh"] > 0)), 'alert_high_enanti'] = "1"


# ##### Alert 4 - Round Consumption
logger.info("Creating Alert 4 - Round Consumption")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός"))
        & (((df["parousa_endeiksi"] > 0)
            & (df["parousa_endeiksi"] % 1000 == 0))
           | ((df["prohgoumeni_endeiksi"] > 0)
              & (df["prohgoumeni_endeiksi"] % 1000 == 0))
           | ((df["katanalwsh_kwh"] > 0)
              & (df["katanalwsh_kwh"] % 1000 == 0)))
        & (df["katanalwsh_kwh"] > 0)), 'alert_round_numbers'] = "1"


# ##### Alert 5 - Μηδενικές Μετρήσεις
logger.info("Creating Alert 5 - Μηδενικές Μετρήσεις")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός"))
        & ((df["parousa_endeiksi"] == 0)
           | (df["prohgoumeni_endeiksi"] == 0)
           | (df["katanalwsh_kwh"] == 0)
           | ((df["max_metrisi"] - df["min_metrisi"]) == 0))), 'alert_zero_figures'] = "1"



# ##### Alert 6 - Γύρισμα Μετρητή
logger.info("Creating Alert 6 - Γύρισμα Μετρητή")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός"))
        & (df["parousa_endeiksi"] < df["prohgoumeni_endeiksi"])), 'alert_girisma_metriti'] = "1"


# ##### Alert 7 - Αλλαγή Μετρητή
logger.info("Creating Alert 7 - Αλλαγή Μετρητή")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός"))
        & (df['paroxi']==df['paroxi'].shift(1))
        & (df['arithmos_metriti']!=df['arithmos_metriti'].shift(1))
        & df['arithmos_metriti'].notnull()
        & df['arithmos_metriti'].shift(1).notnull()
        & (df['arithmos_metriti']!="00")
        & (df['arithmos_metriti'].shift(1)!="00")), 'alert_allagi_metriti'] = "1"

# ##### Alert 8 - Λάθος Μέτρηση

logger.info("Creating Alert 8 - Λάθος Μέτρηση")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός"))
        & (df["synt_wxv"]*(df["correct_parousa_metrisi"]-df["correct_proigoumeni_metrisi"])!=df["katanalwsh_kwh"])), 'alert_check_consumption'] = "1"



# ##### Alert 9 - Consumption before 2018
logger.info("Creating Alert 9 - Consumption before 2018")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός")
         | (df["bill_type"] == "Τελικός")
         | (df["bill_type"] == "Έναντι") )
        & (df["last_date"].dt.year>=2000)
        & (df["last_date"].dt.year<=2017)
        & (df["previous_date"].dt.year>=2000)
        & (df["previous_date"].dt.year<=2017)), 'alert_old_consumption'] = "1"



# ##### Alert 10 - Zero Consumption

logger.info("Creating Alert 10 - Zero Consumption")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός") | (df["bill_type"] == "Τελικός"))
        & (df["katanalwsh_kwh"]==0)), 'alert_zero_consumption'] = "1"


# ##### Alert 11 - Low Consumption
logger.info("Creating Alert 11 - Low Consumption")
df.loc[(((df["bill_type"] == "Εκκαθαριστικός") | (df["bill_type"] == "Τελικός"))
        & (df["perc_consumption"]<=0.4) & (df["perc_consumption"]>0)
        & (df["katanalwsh_kwh"]>0)), 'alert_low_consumption'] = "1"


# ##### Alert 12 - Αλλαγή Παρόχου
logger.info("Creating Alert 12 - Αλλαγή Παρόχου")
df.loc[((df["paroxi"]==df["paroxi"].shift(1))
        & (df["paroxi"].shift(2)==df["paroxi"].shift(1))
        & (df["paroxos"]!=df["paroxos"].shift(1))
        & (df["paroxos"].shift(2)==df["paroxos"].shift(1))
        & (df["bill_type"] != "Έκτακτος")
        & (df["bill_type"].shift(1) != "Έκτακτος")
        & (df["bill_type"].shift(2) != "Έκτακτος")), 'alert_change_provider'] = "1"

df.loc[((df["alert_change_provider"]== "1")
        & (df["paroxi"].shift(1)==df["paroxi"].shift(-1))
        & (df["paroxos"].shift(1)==df["paroxos"].shift(-1))), 'alert_change_provider'] = ""

15-05-2026 11:12:42 INFO [1513941275.<module>:5]: Creating Alert 1 - Μεγάλη Καθυστέρηση του τελευταίου Εκκαθαριστικού
15-05-2026 11:13:14 INFO [1513941275.<module>:28]: Creating Alert 2 - Υψηλός Εκκαθαριστικός
15-05-2026 11:13:34 INFO [1513941275.<module>:53]: Creating Alert 3 - Υψηλός Έναντι
15-05-2026 11:13:34 INFO [1513941275.<module>:62]: Creating Alert 4 - Round Consumption
15-05-2026 11:13:36 INFO [1513941275.<module>:75]: Creating Alert 5 - Μηδενικές Μετρήσεις
15-05-2026 11:13:37 INFO [1513941275.<module>:86]: Creating Alert 6 - Γύρισμα Μετρητή
15-05-2026 11:13:38 INFO [1513941275.<module>:93]: Creating Alert 7 - Αλλαγή Μετρητή
15-05-2026 11:13:41 INFO [1513941275.<module>:105]: Creating Alert 8 - Λάθος Μέτρηση
15-05-2026 11:13:42 INFO [1513941275.<module>:113]: Creating Alert 9 - Consumption before 2018
15-05-2026 11:13:44 INFO [1513941275.<module>:126]: Creating Alert 10 - Zero Consumption
15-05-2026 11:13:45 INFO [1513941275.<module>:132]: Creating Alert 11 - Low Consumption


In [17]:
# ##### Alarms

logger.info("Creating alarms")
df.loc[(df["alert_high_ekkatharistikos"] == "1")
       & (df["perc_consumption"] > 3), 'alarm_priority'] = 4

df.loc[(df["alert_high_ekkatharistikos"] == "1")
       & (df["alert_allagi_metriti"] == "1")
       & (df["perc_consumption"] > 3), 'alarm_priority'] = 3

df.loc[(df["alert_high_ekkatharistikos"] == "1")
       & (df["alert_girisma_metriti"] == "1")
       & (df["perc_consumption"] > 3), 'alarm_priority'] = 2

df.loc[(df["alert_high_ekkatharistikos"] == "1")
       & (df["alert_girisma_metriti"] == "1")
       & (df["alert_allagi_metriti"] == "1")
       & (df["perc_consumption"] > 3), 'alarm_priority'] = 1

df.head()

df.loc[(df["alert_high_enanti"] == "1") & (df["perc_consumption"] > 3), 'alarm_priority'] = 4

df = df.rename(columns={'Date_Bill': 'date_bill', 'days': 'number_of_days2',
                        'minas_teleutaias_katametrisis':'minas_teleutaias_katametrisi'}) 

df['par_dt'] = df.logistiko_date.dt.strftime('%Y%m%d').astype(str)  



# Adjust data types
for column, dtype in type_mapping.items():
    if column in df.columns:
        try:
            if dtype == 'datetime64[ns]':  # Handle datetime separately
                df[column] = pd.to_datetime(df[column], errors='coerce')
            else:
                df[column] = df[column].fillna(0).astype(dtype)
        except Exception as e:
            logger.error(f"Error converting column {column} to {dtype}: {e}")

15-05-2026 11:13:52 INFO [2579590065.<module>:3]: Creating alarms


In [18]:
target_table = 'plpl2_dummy'
schema = 'energy_efficiency'

with vertica_python.connect(**conn_info) as connection:
    with connection.cursor() as cursor:
        try:
            cursor.execute(f"CREATE TABLE IF NOT EXISTS {schema}.{target_table} AS SELECT * FROM {schema}.plpl2;")
            logger.info(f"Table {target_table} created")
        except Exception as e:
            logger.warning(f"Did not create table due to {e}")
            connection.rollback()

15-05-2026 11:15:29 INFO [645792091.<module>:8]: Table plpl2_dummy created


In [19]:
with vertica_python.connect(**conn_info) as connection:
    with connection.cursor() as cursor:       
        try:
            # clear table
            cursor.execute(f"DELETE FROM {schema}.{target_table};")
            connection.commit()
            logger.info("Table data deleted successfully.")
        except QueryError as e:
            logger.error(f"Error deleting data from the table\n{e}")

# Check for data issues BEFORE insertion
null_counts = df.isnull().sum()
if null_counts.any():
    logger.warning(f"Null values: {null_counts[null_counts > 0].to_dict()}")

problematic_chars = []
for col in df.select_dtypes(include=['object']).columns:
    prob_rows = df[df[col].astype(str).str.contains(r'[\t\n\r]', na=False, regex=True)]
    if not prob_rows.empty:
        problematic_chars.append(f"{col}: {len(prob_rows)} rows with tabs/newlines")

if problematic_chars:
    logger.warning(f"Problematic characters: {problematic_chars}")

for col in df.select_dtypes(include=['object']).columns:
    max_len = df[col].astype(str).str.len().max()
    if max_len > 1000:
        logger.warning(f"Very long strings in {col}: max length {max_len}")


15-05-2026 11:15:29 INFO [805116041.<module>:7]: Table data deleted successfully.
15-05-2026 11:15:42 WARNING [805116041.<module>:14]: Null values: {'hmeromhnia_teleutaias_katametrhshs': 106732, 'hmeromhnia_prohgoumenis_katametrhshs': 160239, 'power_off_date': 2445866, 'last_date_ek': 517}


In [20]:
logger.info(f"Writing data to table: {schema}.{target_table}")

# stream DataFrame directly to Vertica
output = io.StringIO()
df.to_csv(output, index=False, na_rep="", date_format="%Y-%m-%d %H:%M:%S")
output.seek(0)

try:
    with vertica_python.connect(**conn_info) as connection:
        with connection.cursor() as cursor:
            cursor.execute(f"DROP TABLE IF EXISTS {schema}.{target_table}_rejected;")
            
            cursor.copy(
                f"""
                COPY {schema}.{target_table}
                FROM STDIN
                PARSER fcsvparser(delimiter=',', header='true')
                REJECTED DATA AS TABLE {schema}.{target_table}_rejected;
                """,
                output)
            connection.commit()
            logger.info(f"Inserted {len(df):,} rows into {schema}.{target_table}")
except Exception as e:
    logger.exception(f"COPY failed: {e}")
    raise

# check rejected rows
rej = vertica_python.connect(**conn_info).cursor().execute(f"select count(*) from {schema}.{target_table}_rejected").fetchall()[0][0]

if rej > 0:
    logger.error(f"{rej} rows rejected - check {schema}.{target_table}_rejected")
else:
    logger.info("No rejected rows.")

15-05-2026 11:18:35 INFO [2467534998.<module>:1]: Writing data to table: energy_efficiency.plpl2_dummy
15-05-2026 11:23:45 INFO [2467534998.<module>:22]: Inserted 2,472,382 rows into energy_efficiency.plpl2_dummy
15-05-2026 11:23:45 INFO [2467534998.<module>:33]: No rejected rows.


## Email

In [21]:
# Build final report text and email it
from utils.send_mail import *

try:
    send_report_text(
        report_text=build_report_text(use_logs=False, logger=logger),
        subject=f"Energy Bills Automation Runtime Report — {datetime.now():%Y-%m-%d %H:%M:%S}",
        recipients=["e.zacharis@msensis.com"],
        attachment_paths=[get_log_report(logger=logger)[0]], 
        logger=logger
    )
    logger.info("Final report emailed.")
except Exception as e:
    logger.exception(f"ERROR sending final report email: {e}")

15-05-2026 11:26:05 INFO [send_mail.get_log_report:31]: Log file found at /home/eyzacharis/Energy Bills/logs
15-05-2026 11:26:05 INFO [send_mail.send_report_text:146]: Sending email to e.zacharis@msensis.com
15-05-2026 11:26:05 INFO [4024296789.<module>:12]: Final report emailed.
